🌌 Astra-LLC — Adaptive Lifelong Learning Companion Multi-Agent Gemini-Powered AI for Personalized Data Science Mastery Track: Concierge Agents Core Idea: A continuously self-evolving multi-agent learning companion that models the learner’s cognitive profile, predicts knowledge decay, adapts curriculum dynamically, and simulates real-world DS interviews.

Astra-LLC goes beyond a tutor. It learns you, adapts to you, and evolves with you — making it the first “Lifelong AI Learning Companion”.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/agents-intensive-capstone-project/Hackathon dataset.txt
/kaggle/input/astra-2-png/astra (1).png


In [2]:
!pip install google-generativeai ipywidgets faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 12.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.3 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but you have pyopenssl 25.3.0 which is incompat

In [3]:
import os, time, json, uuid
import numpy as np
from math import sqrt
from typing import Any, Dict, List

import ipywidgets as widgets
from IPython.display import display, clear_output
import google.generativeai as genai

print("Environment ready.")

Environment ready.


In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GEMINI_API_KEY")


In [5]:
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient

try:
    # 1. Get the key using Kaggle Secrets Client
    user_secrets = UserSecretsClient()
    GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY") # This must match your Label exactly

    # 2. Configure Gemini
    genai.configure(api_key=GEMINI_API_KEY)
    
    # 3. Load Model
    GEMINI_MODEL_NAME = "gemini-1.5-flash"
    gemini_model = genai.GenerativeModel(GEMINI_MODEL_NAME)
    
    print(f"✅ Success! {GEMINI_MODEL_NAME} is connected.")

except Exception as e:
    print("❌ Error: Could not load the key.")
    print("Details:", e)
    print("\nPlease check step 2 below to ensure your Secret is set up correctly.")

✅ Success! gemini-1.5-flash is connected.


In [6]:
def call_gemini(prompt, max_retries=3, delay=1):
    """
    Robust Gemini call:
    - retries on rate limit
    - safe fallback
    - offline mode enabled
    """
    if gemini_model is None:
        return "(Offline Mode) MOCK: " + prompt[:200]

    for attempt in range(max_retries):
        try:
            out = gemini_model.generate_content(prompt)
            return out.text
        except Exception as e:
            err = str(e).lower()
            if "rate" in err or "quota" in err or "timeout" in err:
                time.sleep(delay * (attempt + 1))
                continue
            return f"[Gemini-Error] {e}"

    return "(Fallback Mode) Gemini unreachable."


In [7]:
# ==========================================================
# FIXED MERGED SECTIONS 6, 7, 8 (Robust Hybrid Memory + Agents)
# ==========================================================
import os, time, json, uuid
import numpy as np
from dataclasses import dataclass, field
from typing import Any, Dict, List
import google.generativeai as genai

# ================================
# Load Gemini safely
# ================================
if "GEMINI_API_KEY" in os.environ:
    genai.configure(api_key=os.environ["GEMINI_API_KEY"])
    gemini_model = genai.GenerativeModel("gemini-1.5-flash")
else:
    gemini_model = None


# ================================
# Safe Gemini wrapper
# ================================
def call_gemini(prompt, max_retries=3):
    if gemini_model is None:
        return "(Offline Mode) MOCK RESPONSE:\n" + prompt[:200]

    for i in range(max_retries):
        try:
            res = gemini_model.generate_content(prompt)
            return res.text
        except Exception as e:
            if "rate" in str(e).lower():
                time.sleep(1.5)
                continue
            return f"[GeminiError] {e}"
    return "(Fallback) Gemini unreachable."


# ==========================================================
# SECTION 6 — FIXED HYBRID MEMORY
# ==========================================================
try:
    import faiss
    print("FAISS loaded.")
except:
    faiss = None
    print("FAISS NOT AVAILABLE → fallback mode.")


embedding_model = genai.GenerativeModel("models/embedding-001")

def embed_text(text):
    try:
        emb = embedding_model.embed_content(text=text)
        return np.array(emb["embedding"], dtype=np.float32)
    except:
        v = np.zeros(256, dtype=np.float32)
        for i, ch in enumerate(text[:256]):
            v[i] = ord(ch) / 255.0
        return v


class JSONMemoryBank:

    def __init__(self, file="json_memory_bank.json"):
        self.file = file
        try:
            with open(file, "r") as f:
                self.memory = json.load(f)
        except:
            self.memory = {}

    def save(self):
        with open(self.file, "w") as f:
            json.dump(self.memory, f, indent=2)

    def get_user(self, uid):
        return self.memory.get(uid, {
            "profile": {},
            "progress": {},
            "notes": [],
            "quizzes": {},
        })

    def update_user(self, uid, key, val):
        u = self.get_user(uid)
        u[key] = val
        self.memory[uid] = u
        self.save()

    def append_note(self, uid, note):
        u = self.get_user(uid)
        u["notes"].append({"time": time.time(), "note": note})
        self.memory[uid] = u
        self.save()

    def record_quiz(self, uid, topic, score):
        u = self.get_user(uid)
        u["quizzes"][topic] = {"score": score, "time": time.time()}
        self.memory[uid] = u
        self.save()


class VectorMemoryBank:

    def __init__(self):
        self.texts = []
        self.index = None
        self.dim = None

    def _ensure(self, vec):
        dim = len(vec)
        if self.index is None or self.dim != dim:
            self.dim = dim
            if faiss:
                self.index = faiss.IndexFlatL2(dim)
            else:
                self.index = None

    def add(self, text):
        v = embed_text(text).astype("float32")
        self._ensure(v)
        self.texts.append(text)
        if self.index:
            self.index.add(v.reshape(1, -1))

    def search(self, query, top_k=5):
        v = embed_text(query).astype("float32")
        self._ensure(v)

        if self.index:
            D, I = self.index.search(v.reshape(1, -1), top_k)
            res = []
            for dist, idx in zip(D[0], I[0]):
                if 0 <= idx < len(self.texts):
                    res.append({"text": self.texts[idx], "distance": float(dist)})
            return res

        # fallback
        sims = []
        for t in self.texts:
            diff = v - embed_text(t)
            d = float(np.dot(diff, diff))
            sims.append({"text": t, "distance": d})
        sims.sort(key=lambda x: x["distance"])
        return sims[:top_k]


class HybridMemory:

    def __init__(self, json_mem, vec_mem):
        self.json = json_mem
        self.vec = vec_mem

    def get_user(self, uid):
        return self.json.get_user(uid)

    def save_note(self, uid, txt):
        self.json.append_note(uid, txt)
        self.vec.add(f"{uid}:{txt}")

    def semantic_search(self, uid, query, top_k=5):
        return self.vec.search(query, top_k)


json_mem = JSONMemoryBank()
vec_mem = VectorMemoryBank()
memory = HybridMemory(json_mem, vec_mem)

print("Hybrid Memory initialized.")


# ==========================================================
# SECTION 7 — AGENTS
# ==========================================================

@dataclass
class AgentResponse:
    text: str
    metadata: dict = field(default_factory=dict)


class BaseAgent:
    def __init__(self, name, memory):
        self.name = name
        self.memory = memory


class PlannerAgent(BaseAgent):

    def respond(self, uid, payload, session):
        prompt = (
            "You are an adaptive DS/ML curriculum planner.\n"
            "Use this JSON payload:\n"
            f"{payload}"
        )
        resp = call_gemini(prompt)
        u = self.memory.get_user(uid)
        prog = u.get("progress", {})
        prog["plan"] = resp
        self.memory.json.update_user(uid, "progress", prog)
        self.memory.save_note(uid, "PLAN_CREATED")
        return AgentResponse(resp)


class TutorAgent(BaseAgent):

    def respond(self, uid, question, session):
        profile = self.memory.get_user(uid)["profile"]
        tone = profile.get("preferred_tone", "adaptive")

        prompt = (
            f"You are a DS/ML tutor. Use tone={tone}.\n"
            "Explain clearly with examples.\n\n"
            f"{question}"
        )
        out = call_gemini(prompt)
        self.memory.save_note(uid, f"TUTOR_QA:{question[:60]}")
        return AgentResponse(out)


class RetrieverAgent(BaseAgent):

    def respond(self, uid, topic, session):
        prompt = f"Give 5 curated resources for DS topic: {topic}"
        resp = call_gemini(prompt)
        self.memory.save_note(uid, f"RESOURCES:{topic}")
        return AgentResponse(resp)


class EvaluatorAgent(BaseAgent):

    def generate(self, uid, topic):
        prompt = f"Create 3 conceptual MCQs for: {topic}"
        resp = call_gemini(prompt)
        self.memory.save_note(uid, f"QUIZ_GEN:{topic}")
        return resp

    def grade(self, uid, topic, answer):
        prompt = (
            "Grade this answer 0–1.\n"
            "Return JSON: {\"score\": float, \"feedback\": \"...\"}\n\n"
            f"{answer}"
        )
        out = call_gemini(prompt)

        try:
            data = json.loads(out)
            score = float(data["score"])
            fb = data["feedback"]
        except:
            score = 0.5
            fb = "Could not parse JSON."

        self.memory.json.record_quiz(uid, topic, score)
        self.memory.save_note(uid, f"QUIZ_GRADE:{topic}:{score}")

        return score, fb


# ==========================================================
# SECTION 8 — COORDINATOR
# ==========================================================
class Coordinator:

    def __init__(self, memory):
        self.memory = memory
        self.planner = PlannerAgent("planner", memory)
        self.tutor = TutorAgent("tutor", memory)
        self.retriever = RetrieverAgent("retriever", memory)
        self.evaluator = EvaluatorAgent("evaluator", memory)

    def create_plan(self, uid, goal, hours, weeks):
        payload = json.dumps({"goal": goal, "hours": hours, "weeks": weeks})
        return self.planner.respond(uid, payload, {})

    def tutor_user(self, uid, question):
        return self.tutor.respond(uid, question, {})

    def get_resources(self, uid, topic):
        return self.retriever.respond(uid, topic, {})

    def create_quiz(self, uid, topic):
        return self.evaluator.generate(uid, topic)

    def grade_quiz(self, uid, topic, answer):
        return self.evaluator.grade(uid, topic, answer)


coord = Coordinator(memory)
print("Coordinator initialized successfully.")


FAISS loaded.
Hybrid Memory initialized.
Coordinator initialized successfully.


In [8]:
# ==========================================================
# SECTION 10 — Cognitive Twin Engine (CTE)
# ==========================================================
import time
import math

class CognitiveTwinEngine:

    def __init__(self, memory):
        self.memory = memory

    # ----------------------------------------------------------------
    # Load or create Cognitive Twin for user
    # ----------------------------------------------------------------
    def load_twin(self, uid):
        twin = self.memory.get(uid, "cognitive_twin")
        if twin is None:
            twin = {
                "topics_mastery": {},
                "mistakes": [],
                "forgetting_curve": {},
                "mood_history": [],
                "last_update": time.time()
            }
            self.memory.update(uid, "cognitive_twin", twin)
        return twin

    # ----------------------------------------------------------------
    # Save twin back to memory
    # ----------------------------------------------------------------
    def save_twin(self, uid, twin):
        self.memory.update(uid, "cognitive_twin", twin)

    # ----------------------------------------------------------------
    # Update topic mastery via quiz score (EWMA)
    # ----------------------------------------------------------------
    def update_from_quiz(self, uid, topic, score):

        twin = self.load_twin(uid)
        mastery = twin["topics_mastery"].get(topic, 0.0)

        # EWMA update
        new_mastery = 0.7 * mastery + 0.3 * score
        twin["topics_mastery"][topic] = round(new_mastery, 4)

        # Log mistakes
        if score < 0.6:
            twin["mistakes"].append({
                "topic": topic,
                "score": score,
                "timestamp": time.time()
            })

        self.save_twin(uid, twin)

    # ----------------------------------------------------------------
    # Tutor signals: confusion, curiosity, etc.
    # ----------------------------------------------------------------
    def update_from_tutor(self, uid, topic, mood):
        twin = self.load_twin(uid)

        if mood == "confused":
            # Lower mastery slightly
            if topic in twin["topics_mastery"]:
                twin["topics_mastery"][topic] *= 0.95

        twin["mood_history"].append({
            "topic": topic,
            "mood": mood,
            "timestamp": time.time()
        })

        self.save_twin(uid, twin)

    # ----------------------------------------------------------------
    # Forgetting curve — decay mastery over time
    # ----------------------------------------------------------------
    def apply_forgetting(self, uid):
        twin = self.load_twin(uid)

        now = time.time()
        days_passed = (now - twin["last_update"]) / (60 * 60 * 24)
        twin["last_update"] = now

        if days_passed <= 0:
            return

        # Decay mastery using exponential forgetting
        for topic in twin["topics_mastery"]:
            twin["topics_mastery"][topic] *= (0.99 ** days_passed)

        self.save_twin(uid, twin)

    # ----------------------------------------------------------------
    # Gemini Summary of Learner Brain
    # ----------------------------------------------------------------
    def summarize_cognition(self, uid):
        twin = self.load_twin(uid)
        text = f"""
        Cognitive summary for {uid}:
        
        Mastery Levels:
        {twin['topics_mastery']}

        Mistakes (weak areas):
        {twin['mistakes'][-3:]}

        Recent moods:
        {twin['mood_history'][-5:]}
        """

        if gemini_model:
            return gemini_model.generate_content(
                f"Turn this cognitive learner data into a structured, clear summary:\n{text}"
            ).text

        return text


# ==========================================================
# Instantiate the engine
# ==========================================================
cognitive_twin = CognitiveTwinEngine(memory)


In [9]:
# ==========================================================
# SECTION 11 — ACE: Autonomic Curriculum Evolution Engine
# ==========================================================
import time

class ACEEngine:
    """
    ACE = Autonomic Curriculum Evolution Engine

    Uses:
      - existing weekly plan (progress['plan'])
      - cognitive twin (topics_mastery, mistakes)
      - quiz history

    to generate an UPDATED / EVOLVED plan that:
      - focuses on weak areas
      - adds revision blocks
      - adapts to strategy: balanced / revision / projects / interview
    """

    def __init__(self, memory):
        self.memory = memory

    def _get_user_state(self, uid: str):
        user = self.memory.get_user(uid)
        progress = user.get("progress", {})
        current_plan = progress.get("plan", "No existing plan defined yet.")

        twin = user.get("cognitive_twin", {})
        topics_mastery = twin.get("topics_mastery", {})
        mistakes = twin.get("mistakes", [])
        quizzes = user.get("quizzes", {})

        return user, progress, current_plan, topics_mastery, mistakes, quizzes

    def evolve_plan(
        self,
        uid: str,
        strategy: str = "balanced",
        horizon_weeks: int | None = None
    ) -> str:
        """
        Create an evolved version of the learning plan.

        strategy ∈ {"balanced", "revision", "projects", "interview"}
        horizon_weeks: optional override for number of weeks to plan for.
        """
        user, progress, current_plan, topics_mastery, mistakes, quizzes = \
            self._get_user_state(uid)

        if horizon_weeks is None:
            # Try to infer horizon from old plan; fallback to 8
            horizon_weeks = 8

        # Build a rich prompt for Gemini
        prompt = f"""
You are an adaptive curriculum designer for Data Science & ML.

Your job:
- Take the CURRENT learning plan.
- Read the LEARNER MODEL (cognitive twin).
- Read quiz performance data.
- Produce an EVOLVED plan for the next {horizon_weeks} weeks.

The new plan must:
- Be structured week by week (Week 1, Week 2, ...).
- Emphasize weak topics (low mastery, frequent mistakes).
- Insert explicit REVISION sessions for decayed/forgotten topics.
- Respect strategy mode = "{strategy}" (see rules below).
- Stay realistic in workload (assume 6–12 hours per week unless specified).

STRATEGY MODES:
- "balanced": mix of new concepts, practice, and revision.
- "revision": focus mainly on reinforcing weak / decayed topics.
- "projects": prioritize practical projects, Kaggle-style work, case-studies.
- "interview": prioritize topics & drills that prepare for DS/ML interviews.

CURRENT PLAN:
{current_plan}

LEARNER MODEL — topics_mastery (0–1, lower = weaker):
{topics_mastery}

LEARNER MODEL — mistakes (recent):
{mistakes}

QUIZ HISTORY:
{quizzes}

Now, generate ONLY the new evolved weekly plan,
with clear headings and bullet points per week.
Highlight how the plan adapts to this learner (e.g. "extra revision on X").
"""

        new_plan = call_gemini(prompt)

        # Fallback if Gemini is offline
        if not isinstance(new_plan, str) or new_plan.strip() == "" or \
           new_plan.startswith("(Offline"):
            new_plan = (
                "ACE could not generate a new plan (offline mode).\n\n"
                "Reusing the existing plan:\n\n" + current_plan
            )

        # Store evolved plan & timestamp
        progress["evolved_plan"] = new_plan
        progress["evolved_plan_meta"] = {
            "strategy": strategy,
            "horizon_weeks": horizon_weeks,
            "updated_at": time.time(),
        }
        self.memory.json.update_user(uid, "progress", progress)
        self.memory.save_note(uid, f"ACE_EVOLVED_PLAN_{strategy.upper()}")

        return new_plan

    def explain_changes(self, uid: str) -> str:
        """
        Optional: Ask Gemini to explain what's different between
        the original plan and the evolved plan.
        """
        user = self.memory.get_user(uid)
        progress = user.get("progress", {})
        old = progress.get("plan", "")
        new = progress.get("evolved_plan", "")

        if not old or not new:
            return ("No comparison available. Make sure you created an initial "
                    "plan and then called evolve_plan() at least once.")

        prompt = f"""
You are a helpful coach.

Compare these two learning plans for the same learner:

ORIGINAL PLAN:
{old}

EVOLVED PLAN:
{new}

Explain in a concise way:
- What has changed
- Which weak topics are now emphasized
- How revision is scheduled
- How this should help the learner

Use bullet points and keep it under ~250 words.
"""
        explanation = call_gemini(prompt)

        if not isinstance(explanation, str) or explanation.strip() == "":
            explanation = "Could not summarize changes (offline mode)."

        self.memory.save_note(uid, "ACE_EXPLAINED_CHANGES")
        return explanation


# Instantiate ACE engine
ace_engine = ACEEngine(memory)
print("ACE Engine initialized.")


ACE Engine initialized.


In [10]:
import os

for f in os.listdir("/kaggle/input/astra-2-png"):
    print(f)


astra (1).png


In [11]:
from IPython.display import HTML, display
import base64

# --- 1. Read logo file and convert to base64 ---
logo_file_path = "/kaggle/input/astra-2-png/astra (1).png"  # exact path
with open(logo_file_path, "rb") as f:
    logo_b64 = base64.b64encode(f.read()).decode("utf-8")

logo_data_url = f"data:image/png;base64,{logo_b64}"

# --- 2. Clean, sharp, non-blurry theme (V4) ---
css_v4 = """
<style>
body.jp-Notebook {
    background: radial-gradient(circle at top, #232a47 0%, #111528 45%, #080a14 100%) !important;
    color: #f5f7ff !important;
    font-family: 'Segoe UI', system-ui, sans-serif !important;
}

/* Main app shell */
.astra-main {
    width: 100%;
    max-width: 1350px;
    margin: 25px auto;
    padding: 32px 36px 40px 36px;
    border-radius: 26px;
    background: #181f3a;
    border: 1px solid #3b4d88;
    box-shadow: 0 18px 40px rgba(0, 0, 0, 0.55);
}

/* Header: more height, no cutting, strong contrast */
.astra-header {
    text-align: center;
    padding: 18px 0 28px 0;
    margin-bottom: 20px;
    border-bottom: 1px solid rgba(130, 155, 235, 0.45);
}

/* Logo – now from base64, we just style it */
.astra-logo {
    width: 210px;
    margin-bottom: 10px;
    display: inline-block;
    filter: drop-shadow(0 0 20px rgba(110, 168, 255, 0.8));
}

/* Titles */
.main-title {
    font-size: 44px;
    font-weight: 900;
    margin: 0;
    background: linear-gradient(135deg, #7da8ff, #c491ff);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    letter-spacing: 3px;
}

.company-llc {
    font-size: 22px;
    margin: 4px 0 4px 0;
    color: #d5defa;
    letter-spacing: 2px;
}

.tagline {
    font-size: 14px;
    margin: 0;
    color: #9fb2ff;
}

/* User ID field row */
.user-row {
    margin: 16px 0 8px 0;
}

/* Generic input styling */
.astra-input {
    background: #202954 !important;
    border-radius: 12px !important;
    border: 2px solid #3d5392 !important;
    color: #f1f4ff !important;
    padding: 10px 14px !important;
    font-size: 14px !important;
}

/* Bigger goal & question textareas */
textarea.astra-input {
    height: 110px !important;
}

/* Tabs */
.astra-tabs .p-TabBar .p-TabBar-tab {
    background: #222b54 !important;
    border-radius: 12px 12px 0 0 !important;
    padding: 10px 20px !important;
    margin-right: 4px !important;
    color: #d8e1ff !important;
    font-size: 13px !important;
}

.astra-tabs .p-TabBar .p-TabBar-tab.p-mod-current {
    background: linear-gradient(135deg, #6f94ff, #b480ff) !important;
    color: #ffffff !important;
    box-shadow: 0 6px 16px rgba(110, 168, 255, 0.45);
}

/* Tab content card */
.tab-content {
    background: #1b2243;
    border-radius: 18px;
    padding: 22px 20px 24px 20px;
    border: 1px solid #344475;
    margin-top: 10px;
}

/* Labels for sliders look clearer */
.widget-label {
    color: #cfd7ff !important;
    font-weight: 600 !important;
    font-size: 13px !important;
}

/* Outputs */
.astra-output {
    background: #151b34;
    border-radius: 14px;
    border: 1px solid #313f70;
    padding: 18px;
    font-size: 13px;
    margin-top: 16px;
    min-height: 150px;
    max-height: 320px;
    overflow-y: auto;
}

/* Buttons */
.cosmic-btn {
    width: 100%;
    padding: 12px;
    border-radius: 14px;
    border: none;
    background: linear-gradient(135deg, #6f94ff, #c580ff);
    color: white !important;
    font-weight: 700;
    font-size: 13px;
    margin-top: 16px;
    cursor: pointer;
    transition: transform 0.2s ease, box-shadow 0.2s ease;
}

.cosmic-btn:hover {
    transform: translateY(-1px);
    box-shadow: 0 0 14px rgba(140, 175, 255, 0.7);
}
</style>
"""

display(HTML(css_v4))
print("✅ ASTRA V4 CSS + Logo data ready.")


✅ ASTRA V4 CSS + Logo data ready.


In [12]:
import ipywidgets as widgets
from IPython.display import display

# ---------- safety: coordinator must exist ----------
try:
    coord
    memory
except NameError:
    raise RuntimeError("Run Sections 6–8 first to create coord + memory.")


# ---------- create a user ----------
uid = "shubham_" + uuid.uuid4().hex[:6]
memory.json.update_user(uid, "profile", {
    "username": "Shubham",
    "level": "beginner",
    "preferred_tone": "adaptive"
})

# ---------- widgets ----------
user_box = widgets.Text(
    value=uid,
    description="User ID:",
    layout=widgets.Layout(width="60%")
)
user_box.add_class("astra-input")

# Planner
goal_box = widgets.Textarea(
    value="Become job-ready in Data Science with hands-on projects.",
    description="Goal:",
    layout=widgets.Layout(width="100%")
)
goal_box.add_class("astra-input")

hours_slider = widgets.IntSlider(
    value=15, min=5, max=40, step=5,
    description="Hours/week:",
    layout=widgets.Layout(width="70%")
)
weeks_slider = widgets.IntSlider(
    value=12, min=4, max=52, step=4,
    description="Weeks:",
    layout=widgets.Layout(width="70%")
)

plan_btn = widgets.Button(description="🚀 Generate Learning Plan")
plan_btn.add_class("cosmic-btn")
plan_out = widgets.Output()
plan_out.add_class("astra-output")

# Tutor
question_box = widgets.Textarea(
    value="Explain regularization and why it helps avoid overfitting.",
    description="Question:",
    layout=widgets.Layout(width="100%")
)
question_box.add_class("astra-input")

tutor_btn = widgets.Button(description="🎓 Ask Tutor")
tutor_btn.add_class("cosmic-btn")
tutor_out = widgets.Output()
tutor_out.add_class("astra-output")

# Resources
topic_box = widgets.Text(
    value="feature engineering, sklearn pipelines",
    description="Topic:",
    layout=widgets.Layout(width="100%")
)
topic_box.add_class("astra-input")

res_btn = widgets.Button(description="🔍 Get Resources")
res_btn.add_class("cosmic-btn")
res_out = widgets.Output()
res_out.add_class("astra-output")

# Quiz
quiz_topic_box = widgets.Text(
    value="model evaluation metrics",
    description="Quiz Topic:",
    layout=widgets.Layout(width="100%")
)
quiz_topic_box.add_class("astra-input")

quiz_btn = widgets.Button(description="🧠 Create Quiz")
quiz_btn.add_class("cosmic-btn")
quiz_out = widgets.Output()
quiz_out.add_class("astra-output")

answer_box = widgets.Textarea(
    value="", description="Answer:",
    layout=widgets.Layout(width="100%")
)
answer_box.add_class("astra-input")

grade_btn = widgets.Button(description="📊 Grade Answer")
grade_btn.add_class("cosmic-btn")
grade_out = widgets.Output()
grade_out.add_class("astra-output")


# ---------- callbacks ----------
def on_plan(_):
    with plan_out:
        plan_out.clear_output()
        print(coord.create_plan(user_box.value, goal_box.value,
                                hours_slider.value, weeks_slider.value).text)

def on_tutor(_):
    with tutor_out:
        tutor_out.clear_output()
        print(coord.tutor_user(user_box.value, question_box.value).text)

def on_res(_):
    with res_out:
        res_out.clear_output()
        print(coord.get_resources(user_box.value, topic_box.value).text)

def on_quiz(_):
    with quiz_out:
        quiz_out.clear_output()
        print(coord.create_quiz(user_box.value, quiz_topic_box.value))

def on_grade(_):
    with grade_out:
        grade_out.clear_output()
        score, fb = coord.grade_quiz(user_box.value, quiz_topic_box.value, answer_box.value)
        print(f"SCORE: {score*100:.1f}%")
        print("FEEDBACK:", fb)

plan_btn.on_click(on_plan)
tutor_btn.on_click(on_tutor)
res_btn.on_click(on_res)
quiz_btn.on_click(on_quiz)
grade_btn.on_click(on_grade)


# ---------- header with base64 logo ----------
header_html = f"""
<div class="astra-header">
  <img src="{logo_data_url}" class="astra-logo"/>
  <h1 class="main-title">ASTRA LLC</h1>
  <h3 class="company-llc">LIFELONG COMPANION</h3>
  <p class="tagline">Adaptive Intelligence • Continuous Growth • Cosmic Learning</p>
</div>
"""
header = widgets.HTML(header_html)

# ---------- tabs ----------
planner_tab = widgets.VBox([
    goal_box,
    hours_slider,
    weeks_slider,
    plan_btn,
    plan_out
])
planner_tab.add_class("tab-content")

tutor_tab = widgets.VBox([
    question_box,
    tutor_btn,
    tutor_out
])
tutor_tab.add_class("tab-content")

resources_tab = widgets.VBox([
    topic_box,
    res_btn,
    res_out
])
resources_tab.add_class("tab-content")

quiz_tab = widgets.VBox([
    quiz_topic_box,
    quiz_btn,
    quiz_out,
    answer_box,
    grade_btn,
    grade_out
])
quiz_tab.add_class("tab-content")

tabs = widgets.Tab(children=[planner_tab, tutor_tab, resources_tab, quiz_tab])
tabs.set_title(0, "📅 Planner")
tabs.set_title(1, "🎓 Tutor")
tabs.set_title(2, "📚 Resources")
tabs.set_title(3, "🧠 Quiz")
tabs.add_class("astra-tabs")

ui = widgets.VBox([header,
                   widgets.HBox([user_box], layout=widgets.Layout(width="100%", margin="0 0 10px 0")),
                   tabs])
ui.add_class("astra-main")

display(ui)
print("✨ ASTRA LLC V4 UI Loaded (logo via base64, no blur, higher contrast).")


✨ ASTRA LLC V4 UI Loaded (logo via base64, no blur, higher contrast).


In [13]:
# ==========================================================
# SECTION 11 — ASTRA Meta-Intelligence Layer (A+B+C+D)
#  - A: CognitiveEngine      → learner brain model
#  - B: WorkflowOrchestrator → next-action suggestions
#  - C: ProgressAnalytics    → stats for dashboards
#  - D: DebateEngine         → multi-agent plan refinement
# ==========================================================

import time
from dataclasses import dataclass

# Safety: make sure base system exists
try:
    coord
    memory
except NameError:
    raise RuntimeError("Run Sections 6–8 (memory + coordinator) before Section 11.")


# ==========================
# A) Cognitive Engine
# ==========================
class CognitiveEngine:
    """
    Tracks topic mastery, forgetting, and confusion across time.

    Stores under user['cognitive_profile']:
      {
        "topics": {
            "<topic>": {
                "mastery": float 0–1,
                "last_seen": timestamp,
                "n_quizzes": int,
                "n_confused_signals": int
            },
            ...
        },
        "history": [
            { "timestamp": ..., "event": "quiz", "topic": "...", "score": ... },
            ...
        ]
      }
    """

    def __init__(self, memory):
        self.memory = memory

    def _get_profile(self, uid: str):
        user = self.memory.get_user(uid)
        prof = user.get("cognitive_profile", {"topics": {}, "history": []})
        return prof

    def _save_profile(self, uid: str, profile: dict):
        self.memory.json.update_user(uid, "cognitive_profile", profile)

    def _decay_mastery(self, topic_state: dict, now: float):
        """Apply simple exponential forgetting based on days since last_seen."""
        last_seen = topic_state.get("last_seen", now)
        mastery = float(topic_state.get("mastery", 0.0))
        days = (now - last_seen) / (60 * 60 * 24)
        if days <= 0:
            return mastery
        # 1% decay per day
        return mastery * (0.99 ** days)

    def update_from_quiz(self, uid: str, topic: str, score: float):
        """
        Update mastery based on quiz score (EWMA + forgetting decay).
        score ∈ [0,1]
        """
        now = time.time()
        prof = self._get_profile(uid)
        tstate = prof["topics"].get(topic, {
            "mastery": 0.0,
            "last_seen": now,
            "n_quizzes": 0,
            "n_confused_signals": 0,
        })

        # Apply forgetting to bring mastery to current time
        mastery = self._decay_mastery(tstate, now)

        # Exponential weighted moving average
        alpha = 0.35   # weight of new quiz
        new_mastery = (1 - alpha) * mastery + alpha * float(score)

        tstate["mastery"] = max(0.0, min(1.0, new_mastery))
        tstate["last_seen"] = now
        tstate["n_quizzes"] = int(tstate.get("n_quizzes", 0)) + 1
        prof["topics"][topic] = tstate

        # Log history
        prof["history"].append({
            "timestamp": now,
            "event": "quiz",
            "topic": topic,
            "score": float(score),
        })

        self._save_profile(uid, prof)

    def update_from_tutor(self, uid: str, topic: str, signal: str):
        """
        signal ∈ {"confused", "clear", "curious", ...}
        'confused' slightly lowers mastery; 'clear' slightly raises.
        """
        now = time.time()
        prof = self._get_profile(uid)
        tstate = prof["topics"].get(topic, {
            "mastery": 0.0,
            "last_seen": now,
            "n_quizzes": 0,
            "n_confused_signals": 0,
        })

        mastery = self._decay_mastery(tstate, now)

        if signal == "confused":
            mastery *= 0.9
            tstate["n_confused_signals"] = int(tstate.get("n_confused_signals", 0)) + 1
        elif signal == "clear":
            mastery = min(1.0, mastery + 0.05)

        tstate["mastery"] = mastery
        tstate["last_seen"] = now
        prof["topics"][topic] = tstate

        prof["history"].append({
            "timestamp": now,
            "event": "tutor_signal",
            "topic": topic,
            "signal": signal,
        })

        self._save_profile(uid, prof)

    def summarize(self, uid: str) -> str:
        """
        Ask Gemini to summarize strengths/weaknesses based on cognitive_profile.
        """
        prof = self._get_profile(uid)
        if not prof["topics"]:
            return "No cognitive data yet. Do a few quizzes or tutor sessions first."

        text = json.dumps(prof, indent=2)

        prompt = f"""
You are an expert learning scientist.

A learner has this cognitive profile (JSON):

{text}

Write a concise summary:
- Top 3 strongest topics
- Top 3 weakest topics
- Signs of confusion
- Recommended focus for the next 2 weeks

Keep it under ~250 words, bullet points where useful.
"""
        out = call_gemini(prompt)
        return out

cognitive_engine = CognitiveEngine(memory)


# ==========================
# B) Workflow Orchestrator
# ==========================
class WorkflowOrchestrator:
    """
    Turns high-level intent into ordered steps and tools to call.
    (For now: returns structured text; later can dispatch real tools.)
    """

    def __init__(self, memory):
        self.memory = memory

    def suggest_next_actions(self, uid: str, intent: str) -> str:
        user = self.memory.get_user(uid)
        prof = user.get("cognitive_profile", {})
        plan = user.get("progress", {}).get("plan", "")

        prompt = f"""
You are the Workflow Orchestrator for an AI learning companion.

User intent:
{intent}

Current weekly plan:
{plan}

Cognitive profile:
{prof}

Return a numbered checklist of 3–7 concrete next actions
for the learner to execute in the next 48 hours. Include:
- which agent should handle each step (planner, tutor, quiz, resources)
- whether it's study, practice, or project work
- approximate time cost per item.

Format response as clean text with headings.
"""
        return call_gemini(prompt)

workflow_orchestrator = WorkflowOrchestrator(memory)


# ==========================
# C) Progress Analytics
# ==========================
class ProgressAnalytics:
    """
    Computes simple stats (for future dashboard plots).
    """

    def __init__(self, memory):
        self.memory = memory

    def basic_stats(self, uid: str) -> dict:
        user = self.memory.get_user(uid)
        quizzes = user.get("quizzes", {})
        prof = user.get("cognitive_profile", {})

        scores = [v["score"] for v in quizzes.values()] if quizzes else []
        avg_score = float(sum(scores) / len(scores)) if scores else None

        mastered = []
        weak = []
        topics = prof.get("topics", {})
        for t, s in topics.items():
            m = float(s.get("mastery", 0.0))
            if m >= 0.75:
                mastered.append((t, m))
            elif m <= 0.4:
                weak.append((t, m))

        return {
            "num_quizzes": len(quizzes),
            "avg_score": avg_score,
            "num_topics_tracked": len(topics),
            "mastered_topics": mastered,
            "weak_topics": weak,
        }

    def analytics_text(self, uid: str) -> str:
        stats = self.basic_stats(uid)
        lines = []
        lines.append(f"Quizzes taken: {stats['num_quizzes']}")
        if stats["avg_score"] is not None:
            lines.append(f"Average quiz score: {stats['avg_score']*100:.1f}%")
        lines.append(f"Tracked topics in cognitive engine: {stats['num_topics_tracked']}")

        if stats["mastered_topics"]:
            top = ", ".join(f"{t} ({m:.2f})" for t, m in stats["mastered_topics"][:5])
            lines.append(f"Strong topics: {top}")
        if stats["weak_topics"]:
            low = ", ".join(f"{t} ({m:.2f})" for t, m in stats["weak_topics"][:5])
            lines.append(f"Weak topics: {low}")

        return "\n".join(lines)

progress_analytics = ProgressAnalytics(memory)


# ==========================
# D) Debate Engine (Plan Refiner)
# ==========================
class DebateEngine:
    """
    Uses a virtual Planner vs Critic debate (inside Gemini) to refine plans.
    """

    def __init__(self, memory, coord):
        self.memory = memory
        self.coord = coord

    def refine_plan(self, uid: str) -> str:
        user = self.memory.get_user(uid)
        prof = user.get("cognitive_profile", {})
        plan = user.get("progress", {}).get("plan", "")

        prompt = f"""
Simulate a conversation between two experts:

Planner: designs data science learning plans.
Critic: points out flaws, overload, and missing fundamentals.

They are given:

CURRENT PLAN:
{plan}

COGNITIVE PROFILE:
{prof}

They will have a short 3-round debate, then produce a REFINED PLAN
that is better balanced, more realistic, and focused on weak topics.

Output format:
- Short bullet summary of key changes
- Then the refined weekly plan only.
"""
        refined = call_gemini(prompt)

        # Save back into progress
        progress = user.get("progress", {})
        progress["refined_plan_debate"] = refined
        self.memory.json.update_user(uid, "progress", progress)
        self.memory.save_note(uid, "DEBATE_REFINED_PLAN")

        return refined

debate_engine = DebateEngine(memory, coord)

print("✅ Section 11 meta-intelligence layer initialized:")
print("   - cognitive_engine")
print("   - workflow_orchestrator")
print("   - progress_analytics")
print("   - debate_engine")


✅ Section 11 meta-intelligence layer initialized:
   - cognitive_engine
   - workflow_orchestrator
   - progress_analytics
   - debate_engine


In [14]:
# ==========================================================
# SECTION 12 — INSIGHTS & EVOLUTION DASHBOARD
# ==========================================================
from IPython.display import HTML
import ipywidgets as widgets

# Safety check
try:
    cognitive_engine
    workflow_orchestrator
    progress_analytics
    debate_engine
except NameError:
    raise RuntimeError("Run Section 11 BEFORE Section 12.")


# ==========================================================
# WIDGETS
# ==========================================================

insights_output = widgets.Output()
insights_output.add_class("astra-output")

insights_actions_output = widgets.Output()
insights_actions_output.add_class("astra-output")

insights_plan_output = widgets.Output()
insights_plan_output.add_class("astra-output")

# Buttons
btn_summary = widgets.Button(
    description="🧠 View Cognitive Summary",
    layout=widgets.Layout(width="100%", height="60px")
)
btn_summary.add_class("cosmic-btn-primary")

btn_analytics = widgets.Button(
    description="📊 View Progress Analytics",
    layout=widgets.Layout(width="100%", height="60px")
)
btn_analytics.add_class("cosmic-btn-secondary")

intent_box = widgets.Textarea(
    value="I want to improve my understanding of feature engineering and ML model selection.",
    description="🔗 Intent:",
    layout=widgets.Layout(width="100%", height="90px"),
    style={"description_width": "80px"}
)
intent_box.add_class("astra-input")

btn_next_actions = widgets.Button(
    description="🚀 Generate Next Actions",
    layout=widgets.Layout(width="100%", height="60px")
)
btn_next_actions.add_class("cosmic-btn-primary")

btn_refine_plan = widgets.Button(
    description="🔮 Refine My Learning Plan (Debate Engine)",
    layout=widgets.Layout(width="100%", height="60px")
)
btn_refine_plan.add_class("cosmic-btn-secondary")


# ==========================================================
# CALLBACKS
# ==========================================================

def on_summary_clicked(btn):
    with insights_output:
        insights_output.clear_output()
        uid = user_id_box.value.strip()
        print("🧠 Generating cognitive summary...\n")
        out = cognitive_engine.summarize(uid)
        print(out)

def on_analytics_clicked(btn):
    with insights_output:
        insights_output.clear_output()
        uid = user_id_box.value.strip()
        print("📊 Progress analytics\n")
        out = progress_analytics.analytics_text(uid)
        print(out)

def on_next_actions_clicked(btn):
    with insights_actions_output:
        insights_actions_output.clear_output()
        uid = user_id_box.value.strip()
        intent = intent_box.value.strip()
        print("🚀 Suggesting next steps...\n")
        out = workflow_orchestrator.suggest_next_actions(uid, intent)
        print(out)

def on_refine_plan_clicked(btn):
    with insights_plan_output:
        insights_plan_output.clear_output()
        uid = user_id_box.value.strip()
        print("🔮 Refining plan using Planner–Critic debate...\n")
        out = debate_engine.refine_plan(uid)
        print(out)


btn_summary.on_click(on_summary_clicked)
btn_analytics.on_click(on_analytics_clicked)
btn_next_actions.on_click(on_next_actions_clicked)
btn_refine_plan.on_click(on_refine_plan_clicked)


# ==========================================================
# INSIGHTS TAB LAYOUT
# ==========================================================
insights_tab = widgets.VBox([
    widgets.HTML('<div class="section-header">🔮 ASTRA INSIGHTS & EVOLUTION ENGINE</div>'),

    widgets.HTML("<h4>🧠 Cognitive Understanding</h4>"),
    btn_summary,
    btn_analytics,
    insights_output,

    widgets.HTML("<h4>🚀 Workflow Orchestration</h4>"),
    intent_box,
    btn_next_actions,
    insights_actions_output,

    widgets.HTML("<h4>🔮 Plan Refinement</h4>"),
    btn_refine_plan,
    insights_plan_output

], layout=widgets.Layout(padding="20px"))
insights_tab.add_class("tab-content")

# ADD TAB TO EXISTING TAB BAR
tabs.children = list(tabs.children) + [insights_tab]
tabs.set_title(4, "🔮 INSIGHTS")

print("✨ Section 12 — INSIGHTS TAB ADDED SUCCESSFULLY")
display(HTML("<h3 style='color:#8ecaff'>🔮 ASTRA Insights Dashboard Ready</h3>"))


✨ Section 12 — INSIGHTS TAB ADDED SUCCESSFULLY


In [15]:
# ==========================================================
# SECTION 13 — ASTRA VISUAL DASHBOARD (Charts Tab)
# ==========================================================
import matplotlib.pyplot as plt
from io import BytesIO
from IPython.display import Image, display as ipy_display

# Safety: make sure engines exist
try:
    progress_analytics
    cognitive_engine
except NameError:
    raise RuntimeError("Run Section 11 (engines) before Section 13.")
    
# Safety: make sure main tabs exist
try:
    tabs
except NameError:
    raise RuntimeError("Run Section 9 (UI) before Section 13.")


def get_active_uid():
    """
    Resolve current user id from ASTRA UI.
    Supports:
      - user_box  (V4 UI)
      - user_id_box (older UI)
    """
    try:
        return user_box.value.strip()
    except NameError:
        try:
            return user_id_box.value.strip()
        except NameError:
            raise RuntimeError("No user id widget found (user_box / user_id_box).")


def _plot_mastery_bar(uid):
    user = memory.get_user(uid)
    prof = user.get("cognitive_profile", {})
    topics = prof.get("topics", {})

    if not topics:
        return "No cognitive data yet. Take some quizzes or tutor sessions first."

    topic_names = []
    mastery_vals = []
    for t, s in topics.items():
        topic_names.append(t)
        mastery_vals.append(float(s.get("mastery", 0.0)))

    # sort by mastery descending
    sorted_pairs = sorted(zip(topic_names, mastery_vals), key=lambda x: x[1], reverse=True)
    topic_names = [p[0] for p in sorted_pairs][:10]
    mastery_vals = [p[1] for p in sorted_pairs][:10]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(topic_names[::-1], mastery_vals[::-1])
    ax.set_xlim(0, 1)
    ax.set_xlabel("Mastery (0–1)")
    ax.set_title("Topic Mastery (Top 10)")

    buf = BytesIO()
    plt.tight_layout()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return buf


def _plot_quiz_history(uid):
    user = memory.get_user(uid)
    prof = user.get("cognitive_profile", {})
    hist = prof.get("history", [])

    quiz_events = [h for h in hist if h.get("event") == "quiz"]
    if not quiz_events:
        return "No quiz history yet."

    xs = list(range(1, len(quiz_events) + 1))
    ys = [float(e.get("score", 0.0)) for e in quiz_events]

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(xs, ys, marker="o")
    ax.set_ylim(0, 1)
    ax.set_xlabel("Quiz attempt #")
    ax.set_ylabel("Score (0–1)")
    ax.set_title("Quiz Score Timeline")

    buf = BytesIO()
    plt.tight_layout()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return buf


# ---------- Dashboard widgets ----------
dash_output = widgets.Output()
dash_output.add_class("astra-output")

refresh_dash_btn = widgets.Button(
    description="📊 Refresh Dashboard",
    layout=widgets.Layout(width="100%", height="50px")
)
try:
    refresh_dash_btn.add_class("cosmic-btn")
except Exception:
    pass  # in case only cosmic-btn-primary exists


def on_refresh_dashboard(btn):
    with dash_output:
        dash_output.clear_output()
        uid = get_active_uid()
        print(f"📌 User: {uid}\n")

        # Text analytics
        print("=== Basic Analytics ===")
        print(progress_analytics.analytics_text(uid))
        print("\n=== Visuals ===\n")

        # Mastery chart
        buf1 = _plot_mastery_bar(uid)
        if isinstance(buf1, str):
            print(buf1)
        else:
            ipy_display(Image(data=buf1.read()))
            print("\n")

        # Quiz history chart
        buf2 = _plot_quiz_history(uid)
        if isinstance(buf2, str):
            print(buf2)
        else:
            ipy_display(Image(data=buf2.read()))


refresh_dash_btn.on_click(on_refresh_dashboard)

dashboard_tab = widgets.VBox([
    widgets.HTML("<h3 style='margin-top:0;'>📊 Learning Progress Dashboard</h3>"),
    widgets.HTML("<p>Visual view of your mastery and quiz trajectory.</p>"),
    refresh_dash_btn,
    dash_output
])
dashboard_tab.add_class("tab-content")

# append as new tab
tabs.children = list(tabs.children) + [dashboard_tab]
tabs.set_title(len(tabs.children) - 1, "📊 DASHBOARD")

print("✅ Section 13 — Dashboard Tab added.")


✅ Section 13 — Dashboard Tab added.


In [16]:
# ==========================================================
# SECTION 14 — ASTRA AUTO-COACH (Autonomous Cycle)
# ==========================================================
# Uses: cognitive_engine, workflow_orchestrator, debate_engine, progress_analytics

# Safety
try:
    cognitive_engine
    workflow_orchestrator
    debate_engine
    progress_analytics
except NameError:
    raise RuntimeError("Run Section 11 before Section 14.")
    
try:
    tabs
except NameError:
    raise RuntimeError("Run Section 9 (UI) before Section 14.")


def get_active_uid():
    # re-define here for safety
    try:
        return user_box.value.strip()
    except NameError:
        try:
            return user_id_box.value.strip()
        except NameError:
            raise RuntimeError("No user id widget found.")


auto_output = widgets.Output()
auto_output.add_class("astra-output")

intent_auto = widgets.Textarea(
    value="Make me stronger in weak topics and suggest a realistic next 2-day schedule.",
    description="Intent:",
    layout=widgets.Layout(width="100%", height="80px"),
    style={"description_width": "70px"}
)
intent_auto.add_class("astra-input")

run_auto_btn = widgets.Button(
    description="🤖 Run Auto-Coach Cycle",
    layout=widgets.Layout(width="100%", height="55px")
)
try:
    run_auto_btn.add_class("cosmic-btn")
except Exception:
    pass


def on_run_auto(btn):
    with auto_output:
        auto_output.clear_output()
        uid = get_active_uid()
        intent = intent_auto.value.strip()

        print(f"🚀 ASTRA AUTO-COACH CYCLE for user: {uid}\n")

        # 1) Show current lightweight analytics
        print("=== Current Snapshot ===")
        print(progress_analytics.analytics_text(uid))
        print("\n")

        # 2) Refine plan with debate engine
        print("=== 🔮 Refined Plan (Debate Engine) ===\n")
        refined = debate_engine.refine_plan(uid)
        print(refined[:2000])  # avoid flooding output
        print("\n")

        # 3) Suggest next concrete actions
        print("=== 🚀 Next Actions (Workflow Orchestrator) ===\n")
        next_actions = workflow_orchestrator.suggest_next_actions(uid, intent)
        print(next_actions)
        print("\n")

        # 4) Cognitive high-level summary
        print("=== 🧠 Cognitive Summary ===\n")
        summary = cognitive_engine.summarize(uid)
        print(summary)

        print("\n✅ Auto-coach cycle complete. Follow the suggested steps, then come back and run again.")


run_auto_btn.on_click(on_run_auto)

auto_tab = widgets.VBox([
    widgets.HTML("<h3 style='margin-top:0;'>🤖 ASTRA AUTO-COACH</h3>"),
    widgets.HTML("<p>One-click intelligent cycle: refine plan, suggest actions, and summarize your brain state.</p>"),
    intent_auto,
    run_auto_btn,
    auto_output
])
auto_tab.add_class("tab-content")

# add as new tab
tabs.children = list(tabs.children) + [auto_tab]
tabs.set_title(len(tabs.children) - 1, "🤖 AUTO")
print("✅ Section 14 — Auto-Coach Tab added.")


✅ Section 14 — Auto-Coach Tab added.
